# Checagem de OpenAI Batch API

Notebook auxiliar para monitorar jobs de batch **sem bloquear** o `resultado_rubrica-gpt.ipynb`.

Use um **kernel separado** (este notebook) enquanto o outro roda o polling.

**IDs importantes:**
- `batch_rubrica_gpt_part_01.jsonl` → arquivo local de entrada (não é batch_id)
- `file-...` → arquivo enviado à OpenAI
- `batch_...` → **batch_id** do job (aparece como `Job: batch_...` no outro notebook)

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("..") / ".env")
load_dotenv()

api_key = os.getenv("CHAT_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY não encontrada no .env")

client = OpenAI(api_key=api_key)
print("Cliente OpenAI OK | CWD:", os.getcwd())

Cliente OpenAI OK | CWD: c:\Users\anara\Desktop\Pesquisa - Tutoria\feedback-open-source\notebooks


## Status possíveis

| Status | Significado |
|--------|-------------|
| `validating` | OpenAI validando o arquivo de entrada |
| `in_progress` | Processando requests |
| `finalizing` | Consolidando resultados |
| `completed` | Concluído (ver `output_file_id` / `error_file_id`) |
| `failed` | Job falhou inteiro |
| `cancelled` | Cancelado manualmente |
| `expired` | Passou da janela de 24h |

Se `completed=0` e `failed=250` após horas, leia o arquivo de erros na célula abaixo.

In [7]:
LIMITE_LISTAGEM = 10

rows = []
for job in client.batches.list(limit=LIMITE_LISTAGEM).data:
    counts = job.request_counts
    rows.append(
        {
            "batch_id": job.id,
            "status": job.status,
            "created_at": datetime.fromtimestamp(job.created_at, tz=timezone.utc),
            "endpoint": job.endpoint,
            "total": counts.total if counts else None,
            "completed": counts.completed if counts else None,
            "failed": counts.failed if counts else None,
            "output_file_id": job.output_file_id,
            "error_file_id": job.error_file_id,
        }
    )

df_batches = pd.DataFrame(rows)
df_batches

,batch_id,status,created_at,endpoint,total,completed,failed,output_file_id,error_file_id
0,batch_6ab32c066b9c8190beb7b317c84432dd,in_progress,2026-09-23 01:31:50+00:00,/v1/chat/completions,250,0,0,None,None
1,batch_6ab2dcfd22308190bf2a4cec486b7fbe,completed,2026-09-22 19:54:37+00:00,/v1/chat/completions,250,250,0,file-4nEZ7ZrsikYTS4oa8LLtaH,None
2,batch_6ab2d55d984081908ee0605ee0fe1796,completed,2026-09-22 19:22:05+00:00,/v1/chat/completions,250,0,250,None,file-1WX123697TH2PgRQXZVuB4


## Consultar um batch específico

Cole o `batch_id` do log (`Job: batch_...`) ou copie da tabela acima.

In [6]:
BATCH_ID = "batch_6ab2dcfd22308190bf2a4cec486b7fbe"  # ex.: "batch_abc123..."

if not BATCH_ID:
    print("Defina BATCH_ID na célula acima.")
else:
    job = client.batches.retrieve(BATCH_ID)
    counts = job.request_counts

    print("batch_id:", job.id)
    print("status:", job.status)
    print("endpoint:", job.endpoint)
    print("created_at:", datetime.fromtimestamp(job.created_at, tz=timezone.utc))
    print("completed_at:", datetime.fromtimestamp(job.completed_at, tz=timezone.utc) if job.completed_at else None)
    print("input_file_id:", job.input_file_id)
    print("output_file_id:", job.output_file_id)
    print("error_file_id:", job.error_file_id)
    print("request_counts:", counts)

    if counts and counts.total:
        pct = (counts.completed or 0) / counts.total * 100
        print(f"progresso: {counts.completed}/{counts.total} ({pct:.1f}%)")

    if job.metadata:
        print("metadata:", job.metadata)

batch_id: batch_6ab2dcfd22308190bf2a4cec486b7fbe
status: completed
endpoint: /v1/chat/completions
created_at: 2026-09-22 19:54:37+00:00
completed_at: 2026-09-23 01:29:32+00:00
input_file_id: file-BJuUAp8kE7kxyhjGx8RzWn
output_file_id: file-4nEZ7ZrsikYTS4oa8LLtaH
error_file_id: None
request_counts: BatchRequestCounts(completed=250, failed=0, total=250)
progresso: 250/250 (100.0%)
metadata: {'description': 'rubrica-16dim-gpt-part-1'}


## Ler amostra do arquivo de **sucesso** (`output_file_id`)

In [ ]:
N_AMOSTRAS_OUTPUT = 3

if not BATCH_ID:
    print("Defina BATCH_ID primeiro.")
elif not job.output_file_id:
    print("Sem output_file_id — nenhuma request concluiu com sucesso ainda (ou batch não terminou).")
else:
    texto = client.files.content(job.output_file_id).text
    linhas = [ln for ln in texto.splitlines() if ln.strip()]
    print(f"Linhas no output: {len(linhas)}")

    for i, ln in enumerate(linhas[:N_AMOSTRAS_OUTPUT]):
        item = json.loads(ln)
        print(f"\n--- output {i+1} | custom_id={item.get('custom_id')} ---")
        resp = item.get("response") or {}
        print("status_code:", resp.get("status_code"))
        body = (resp.get("body") or {})
        if body.get("choices"):
            content = body["choices"][0]["message"]["content"]
            print("content (início):", content[:400])
        elif body.get("error"):
            print("error:", body["error"])
        else:
            print(json.dumps(item, ensure_ascii=False)[:500])

## Ler amostra do arquivo de **erros** (`error_file_id`)

Útil quando o batch `completed` mas `output_file_id` está vazio.

In [4]:
N_AMOSTRAS_ERRO = 5

if not BATCH_ID:
    print("Defina BATCH_ID primeiro.")
elif not job.error_file_id:
    print("Sem error_file_id — nenhum erro registrado em arquivo separado.")
else:
    texto = client.files.content(job.error_file_id).text
    linhas = [ln for ln in texto.splitlines() if ln.strip()]
    print(f"Linhas no error file: {len(linhas)}")

    erros_resumo = []
    for ln in linhas:
        item = json.loads(ln)
        resp = item.get("response") or {}
        body = resp.get("body") or {}
        err = body.get("error") or item.get("error") or "erro_desconhecido"
        msg = err.get("message") if isinstance(err, dict) else str(err)
        erros_resumo.append({"custom_id": item.get("custom_id"), "mensagem": msg})

    df_erros = pd.DataFrame(erros_resumo)
    print("\nMensagens únicas:")
    print(df_erros["mensagem"].value_counts().head(10))

    print(f"\nPrimeiras {N_AMOSTRAS_ERRO} linhas:")
    display(df_erros.head(N_AMOSTRAS_ERRO))

Sem error_file_id — nenhum erro registrado em arquivo separado.


## Cancelar batch (opcional)

Só execute se quiser interromper um job preso. Mude `CONFIRMAR_CANCELAMENTO = True`.

In [ ]:
CONFIRMAR_CANCELAMENTO = False

if not BATCH_ID:
    print("Defina BATCH_ID primeiro.")
elif not CONFIRMAR_CANCELAMENTO:
    print("Cancelamento desativado. Defina CONFIRMAR_CANCELAMENTO = True para cancelar.")
else:
    cancelled = client.batches.cancel(BATCH_ID)
    print("status após cancel:", cancelled.status)

## Checar progresso local do experimento (CSV + JSON)

Compara o batch na OpenAI com o que já foi salvo pelo `resultado_rubrica-gpt.ipynb`.

In [ ]:
CSV_GPT = Path("..") / "resultado_rubrica" / "prova-tecnologia-results-1500-rubrica-16dim-gpt5mini.csv"
PROGRESSO_JSON = Path("batch_rubrica_gpt_progress.json")

if CSV_GPT.is_file():
    df_csv = pd.read_csv(CSV_GPT)
    print(f"CSV: {CSV_GPT.name}")
    print(f"Linhas salvas (sem header): {len(df_csv)}")
    if "analise_feedback" in df_csv.columns:
        ok = df_csv["analise_feedback"].astype(str).str.contains('"conteudo"', na=False).sum()
        print(f"JSON válidos (heurística): {ok}")
else:
    print(f"CSV ainda não existe: {CSV_GPT}")

if PROGRESSO_JSON.is_file():
    with open(PROGRESSO_JSON, encoding="utf-8") as f:
        prog = json.load(f)
    concluidos = prog.get("concluidos", [])
    print(f"\nProgresso local: {len(concluidos)} índices concluídos")
else:
    print(f"\nSem arquivo de progresso: {PROGRESSO_JSON.name}")